In [1]:
# Import pandas for loading and manipulating the datasets
import pandas as pd

# Load the training features and target
X_train = pd.read_csv("/app/data/X_train.csv")
y_train = pd.read_csv("/app/data/y_train.csv")

# Load the validation features and target
X_val = pd.read_csv("/app/data/X_val.csv")
y_val = pd.read_csv("/app/data/y_val.csv")

# Load the test features and target
X_test = pd.read_csv("/app/data/X_test.csv")
y_test = pd.read_csv("/app/data/y_test.csv")

# Check the dimensions of all datasets
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_val:", X_val.shape)
print("y_val:", y_val.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (67532, 31)
y_train: (67532, 1)
X_val: (14472, 31)
y_val: (14472, 1)
X_test: (14472, 31)
y_test: (14472, 1)


In [3]:
# Define features that contain information unavailable at prediction time
leakage_columns = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "order_status",
    "review_count",
    "average_review_score"
]

In [4]:
# Remove leakage-prone features from all datasets
X_train = X_train.drop(columns=leakage_columns)
X_val = X_val.drop(columns=leakage_columns)
X_test = X_test.drop(columns=leakage_columns)

# Check the remaining number of features after removing leakage
print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

X_train shape: (67532, 26)
X_val shape: (14472, 26)
X_test shape: (14472, 26)


In [5]:
# Convert the order purchase timestamp from text to datetime
# This allows us to extract useful time-based features
for df in [X_train, X_val, X_test]:
    df["order_purchase_timestamp"] = pd.to_datetime(
        df["order_purchase_timestamp"]
    )

In [6]:
# Check that the purchase timestamp was converted to datetime
print(X_train["order_purchase_timestamp"].dtype)

datetime64[us]


In [7]:
# Extract useful time-based features from the order purchase timestamp
for df in [X_train, X_val, X_test]:
    df["purchase_year"] = df["order_purchase_timestamp"].dt.year
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

In [8]:
# Display the newly created time-based features
X_train[
    [
        "purchase_year",
        "purchase_month",
        "purchase_dayofweek",
        "purchase_hour"
    ]
].head()

,purchase_year,purchase_month,purchase_dayofweek,purchase_hour
0,2018,4,1,0
1,2018,6,0,23
2,2018,7,0,20
3,2018,4,2,16
4,2018,8,2,21


In [9]:
# Remove the original timestamp because its useful information
# has already been extracted into separate time-based features
for df in [X_train, X_val, X_test]:
    df.drop(columns=["order_purchase_timestamp"], inplace=True)

In [10]:
# Verify that the original timestamp was removed
print("order_purchase_timestamp" in X_train.columns)

False


In [11]:
# Remove identifier columns because they are unique IDs rather than meaningful predictive features
id_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

for df in [X_train, X_val, X_test]:
    df.drop(columns=id_columns, inplace=True)

# Check the remaining features after removing identifier columns
print("Number of features:", X_train.shape[1])

Number of features: 26


In [12]:
# Identify categorical features that contain text values
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Display the categorical features that will require encoding
categorical_columns

/tmp/ipykernel_511/1131615088.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


['order_approved_at',
 'order_estimated_delivery_date',
 'customer_city',
 'customer_state',
 'payment_types']

In [13]:
# Convert the remaining date columns from text to datetime
# These dates may contain useful information available around the order time
for df in [X_train, X_val, X_test]:
    df["order_approved_at"] = pd.to_datetime(df["order_approved_at"])
    df["order_estimated_delivery_date"] = pd.to_datetime(
        df["order_estimated_delivery_date"]
    )

# Verify that both columns are now stored as datetime values
print(X_train[
    ["order_approved_at", "order_estimated_delivery_date"]
].dtypes)

order_approved_at                datetime64[us]
order_estimated_delivery_date    datetime64[us]
dtype: object


In [15]:
# Reload the original feature datasets to restore the timestamp columns
X_train = pd.read_csv("/app/data/X_train.csv")
X_val = pd.read_csv("/app/data/X_val.csv")
X_test = pd.read_csv("/app/data/X_test.csv")

In [16]:
# Remove features that contain information unavailable at prediction time
leakage_columns = [
    "order_delivered_customer_date",
    "order_delivered_carrier_date",
    "order_status",
    "review_count",
    "average_review_score"
]

for df in [X_train, X_val, X_test]:
    df.drop(columns=leakage_columns, inplace=True)

In [17]:
# Convert order-related date columns to datetime for feature extraction
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date"
]

for df in [X_train, X_val, X_test]:
    for col in date_columns:
        df[col] = pd.to_datetime(df[col])

In [18]:
# Extract useful time features from the purchase timestamp
for df in [X_train, X_val, X_test]:
    df["purchase_year"] = df["order_purchase_timestamp"].dt.year
    df["purchase_month"] = df["order_purchase_timestamp"].dt.month
    df["purchase_dayofweek"] = df["order_purchase_timestamp"].dt.dayofweek
    df["purchase_hour"] = df["order_purchase_timestamp"].dt.hour

# Calculate the time between purchase and order approval
for df in [X_train, X_val, X_test]:
    df["approval_delay_hours"] = (
        df["order_approved_at"] -
        df["order_purchase_timestamp"]
    ).dt.total_seconds() / 3600

# Extract the month of the estimated delivery date
for df in [X_train, X_val, X_test]:
    df["estimated_delivery_month"] = (
        df["order_estimated_delivery_date"].dt.month
    )

In [19]:
# Remove the original datetime columns after extracting the useful information
date_columns_to_drop = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_estimated_delivery_date"
]

for df in [X_train, X_val, X_test]:
    df.drop(columns=date_columns_to_drop, inplace=True)

In [ ]:
# Verify the resulting feature dimensions after feature engineering
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

In [20]:
# Display the current feature names after feature engineering
print(X_train.columns.tolist())

['order_id', 'customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state', 'item_count', 'unique_products', 'unique_sellers', 'total_price', 'total_freight', 'payment_count', 'total_payment', 'payment_types', 'product_count', 'missing_product_category', 'seller_count', 'seller_states', 'category_count', 'geo_lat', 'geo_lng', 'geo_city_count', 'geo_state_count', 'purchase_year', 'purchase_month', 'purchase_dayofweek', 'purchase_hour', 'approval_delay_hours', 'estimated_delivery_month']


In [21]:
# Check that all three datasets have exactly the same features
print("Train/Validation columns match:",
      list(X_train.columns) == list(X_val.columns))

print("Train/Test columns match:",
      list(X_train.columns) == list(X_test.columns))

Train/Validation columns match: True
Train/Test columns match: True


In [22]:
# Check the remaining missing values in the training features
missing_values = X_train.isna().sum()

# Display only features that still contain missing values
missing_values[missing_values > 0].sort_values(ascending=False)

geo_lat                 175
geo_lng                 175
geo_city_count          175
geo_state_count         175
approval_delay_hours     12
dtype: int64

In [23]:
# Identify the numeric features that contain missing values
numeric_missing_columns = [
    "geo_lat",
    "geo_lng",
    "geo_city_count",
    "geo_state_count",
    "approval_delay_hours"
]

# Calculate the median of each feature using only the training data
train_medians = X_train[numeric_missing_columns].median()

# Display the median values that will be used for imputation
train_medians

geo_lat                -22.926257
geo_lng                -46.631370
geo_city_count           1.000000
geo_state_count          1.000000
approval_delay_hours     0.343056
dtype: float64

In [24]:
# Fill missing numeric values using medians calculated from the training set
for df in [X_train, X_val, X_test]:
    df[numeric_missing_columns] = df[numeric_missing_columns].fillna(
        train_medians
    )

In [25]:
# Verify that no missing values remain in the selected numeric features
X_train[numeric_missing_columns].isna().sum()

geo_lat                 0
geo_lng                 0
geo_city_count          0
geo_state_count         0
approval_delay_hours    0
dtype: int64

In [26]:
# Identify the remaining categorical features
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Check the number of unique categories in each categorical feature
for col in categorical_columns:
    print(f"{col}: {X_train[col].nunique()} unique values")

order_id: 67532 unique values
customer_id: 67532 unique values
customer_unique_id: 65929 unique values
customer_city: 3668 unique values
customer_state: 27 unique values
payment_types: 7 unique values


/tmp/ipykernel_511/1603040800.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


In [27]:
# Remove identifier columns because they are unique IDs and do not provide
# meaningful generalizable information for predicting delivery delays
id_columns = [
    "order_id",
    "customer_id",
    "customer_unique_id"
]

for df in [X_train, X_val, X_test]:
    df.drop(columns=id_columns, inplace=True)

# Verify that the identifier columns were removed
print("ID columns remaining:",
      [col for col in id_columns if col in X_train.columns])

ID columns remaining: []


In [28]:
# Identify the remaining categorical features after removing ID columns
categorical_columns = X_train.select_dtypes(
    include=["object"]
).columns.tolist()

# Display the categorical features that require encoding
print(categorical_columns)

['customer_city', 'customer_state', 'payment_types']


/tmp/ipykernel_511/4033049917.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X_train.select_dtypes(


In [29]:
# Calculate how frequently each customer city appears in the training data
# The mapping is learned only from the training set to avoid data leakage
city_frequency = X_train["customer_city"].value_counts(normalize=True)

# Convert each city into its frequency in the training data
for df in [X_train, X_val, X_test]:
    df["customer_city_frequency"] = (
        df["customer_city"]
        .map(city_frequency)
        .fillna(0)
    )

# Remove the original high-cardinality city column
for df in [X_train, X_val, X_test]:
    df.drop(columns=["customer_city"], inplace=True)

In [30]:
# Define the low-cardinality categorical features to encode using one-hot encoding
one_hot_columns = [
    "customer_state",
    "payment_types"
]

# Learn the categories from the training data and create one-hot encoded features
X_train_encoded = pd.get_dummies(
    X_train,
    columns=one_hot_columns,
    dtype=int
)

# Save the encoded column names so validation and test use the exact same features
encoded_columns = X_train_encoded.columns

In [31]:
# Apply the same one-hot encoding to validation and test data
X_val_encoded = pd.get_dummies(
    X_val,
    columns=one_hot_columns,
    dtype=int
)

X_test_encoded = pd.get_dummies(
    X_test,
    columns=one_hot_columns,
    dtype=int
)

# Align validation and test columns with the training feature set
# Missing columns are filled with 0, and extra columns are removed
X_val_encoded = X_val_encoded.reindex(
    columns=encoded_columns,
    fill_value=0
)

X_test_encoded = X_test_encoded.reindex(
    columns=encoded_columns,
    fill_value=0
)

In [32]:
# Replace the original datasets with the aligned encoded datasets
X_train = X_train_encoded
X_val = X_val_encoded
X_test = X_test_encoded

# Check that all datasets now have exactly the same feature columns
print("Train/Validation columns match:", X_train.columns.equals(X_val.columns))
print("Train/Test columns match:", X_train.columns.equals(X_test.columns))

print("Final feature count:", X_train.shape[1])

Train/Validation columns match: True
Train/Test columns match: True
Final feature count: 58


In [33]:
# Check that no missing values remain after feature engineering
print("Missing values in X_train:", X_train.isna().sum().sum())
print("Missing values in X_val:", X_val.isna().sum().sum())
print("Missing values in X_test:", X_test.isna().sum().sum())

Missing values in X_train: 0
Missing values in X_val: 2
Missing values in X_test: 0


In [34]:
# Check that all features are numeric and ready for machine learning models
print("Non-numeric features:", X_train.select_dtypes(exclude="number").columns.tolist())

Non-numeric features: []


In [35]:
# Display the final shapes after feature engineering
print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)

X_train: (67532, 58)
X_val: (14472, 58)
X_test: (14472, 58)
y_train: (67532, 1)
y_val: (14472, 1)
y_test: (14472, 1)


In [36]:
# Verify that no missing values remain in any feature set
print("Missing values:")
print("Train:", X_train.isna().sum().sum())
print("Validation:", X_val.isna().sum().sum())
print("Test:", X_test.isna().sum().sum())

# Verify that all features are numeric and ready for machine learning
print("\nNon-numeric features:")
print(X_train.select_dtypes(exclude="number").columns.tolist())

Missing values:
Train: 0
Validation: 2
Test: 0

Non-numeric features:
[]


In [37]:
# Identify the exact features that still contain missing values in the validation set
validation_missing = X_val.isna().sum()

# Display only validation features that still contain missing values
validation_missing[validation_missing > 0]

payment_count    1
total_payment    1
dtype: int64

In [38]:
# Identify the numeric features that still contain missing values in validation
validation_numeric_missing = [
    "payment_count",
    "total_payment"
]

# Calculate their median values using only the training data
validation_medians = X_train[validation_numeric_missing].median()

# Fill the missing validation values using the training-set medians
X_val[validation_numeric_missing] = X_val[
    validation_numeric_missing
].fillna(validation_medians)

# Verify that the missing values were handled
print(
    X_val[validation_numeric_missing]
    .isna()
    .sum()
)

payment_count    0
total_payment    0
dtype: int64


In [39]:
# Verify that no missing values remain in any dataset
print("Train missing:", X_train.isna().sum().sum())
print("Validation missing:", X_val.isna().sum().sum())
print("Test missing:", X_test.isna().sum().sum())

# Verify that all features are numeric
print(
    "Non-numeric features:",
    X_train.select_dtypes(exclude="number").columns.tolist()
)

# Verify that all datasets have the same feature structure
print(
    "Columns match:",
    X_train.columns.equals(X_val.columns)
    and X_train.columns.equals(X_test.columns)
)

Train missing: 0
Validation missing: 0
Test missing: 0
Non-numeric features: []
Columns match: True


In [40]:
# Save the final engineered feature sets so Notebook 6 can load them directly
X_train.to_csv("/app/data/X_train_engineered.csv", index=False)
X_val.to_csv("/app/data/X_val_engineered.csv", index=False)
X_test.to_csv("/app/data/X_test_engineered.csv", index=False)

# Confirm that the engineered datasets were saved successfully
print("Engineered datasets saved successfully.")

Engineered datasets saved successfully.


In [41]:
# Verify the saved engineered datasets can be loaded correctly
print(pd.read_csv("/app/data/X_train_engineered.csv").shape)
print(pd.read_csv("/app/data/X_val_engineered.csv").shape)
print(pd.read_csv("/app/data/X_test_engineered.csv").shape)

(67532, 58)
(14472, 58)
(14472, 58)
